# Support Vector Machine (SVM) Classifier for EEG Signal Classification

This notebook implements a Support Vector Machine (SVM) classifier for EEG signal classification with comprehensive hyperparameter tuning, cross-validation, and evaluation metrics. The model is designed for binary or multiclass classification tasks on preprocessed EEG data.

## Features:
- Data loading and preprocessing
- Feature standardization
- Hyperparameter tuning using GridSearchCV
- Cross-validation analysis
- Comprehensive evaluation metrics
- Visualization of results

## 1. Import Required Libraries

In [ ]:
# Import standard libraries
import pandas as pd
import numpy as np
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Import scikit-learn modules
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             confusion_matrix, classification_report, roc_auc_score, 
                             roc_curve, auc)
from sklearn.decomposition import PCA

# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec
import time

# Set random seeds for reproducibility
np.random.seed(42)

print("✓ All libraries imported successfully!")

## 2. Data Loading and Configuration

In [ ]:
# Configure file paths - use relative paths for public repository compatibility
# Adjust these paths based on your directory structure

DATA_DIR = "./data"  # Directory containing preprocessed data
MODELS_DIR = "./models"  # Directory to save trained models
RESULTS_DIR = "./results"  # Directory to save results

# Create directories if they don't exist
for directory in [MODELS_DIR, RESULTS_DIR]:
    os.makedirs(directory, exist_ok=True)

# Data file paths
pkl_path = os.path.join(DATA_DIR, 'training_data.pkl')
csv_path = os.path.join(DATA_DIR, 'training_data.csv')

print(f"Data directory: {DATA_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"Results directory: {RESULTS_DIR}")

## 3. Load and Explore Dataset

In [ ]:
# Load the preprocessed dataset
# Try loading from pickle first (faster), then CSV

if os.path.exists(pkl_path):
    with open(pkl_path, 'rb') as f:
        df = pickle.load(f)
    print(f"✓ Loaded dataset from pickle file")
elif os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"✓ Loaded dataset from CSV file")
else:
    raise FileNotFoundError(f"Data file not found at {pkl_path} or {csv_path}")

print(f"\nDataset Shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Samples: {df.shape[0]}")

In [ ]:
# Display dataset information
print("="*80)
print("DATASET OVERVIEW")
print("="*80)

print(f"\nFirst few rows:")
print(df.head())

print(f"\n\nDataset Info:")
print(df.info())

print(f"\n\nClass Distribution:")
class_dist = df['label'].value_counts()
print(class_dist)

if len(class_dist) > 1:
    imbalance_ratio = class_dist.max() / class_dist.min()
    print(f"\nClass Imbalance Ratio (max/min): {imbalance_ratio:.2f}")

print(f"\n\nBasic Statistics:")
print(df.describe())

print(f"\n\nMissing Values:")
missing = df.isnull().sum().sum()
print(f"{missing} missing values found")

## 4. Data Preprocessing

In [ ]:
# Separate features and labels
X = df.drop('label', axis=1).values  # Features
y = df['label'].values  # Labels

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")

# Encode labels if they are strings
if y.dtype == 'object':
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)
    print(f"\nClass encoding: {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")
else:
    label_encoder = None
    print("\nLabels are already numeric")

In [ ]:
# Split data into training and testing sets
test_size = 0.2  # 80% training, 20% testing
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=test_size,
    random_state=random_state,
    stratify=y  # Maintain class distribution
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")
print(f"\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({100*c/len(y_train):.1f}%)")

print(f"\nTesting set class distribution:")
unique, counts = np.unique(y_test, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {u}: {c} samples ({100*c/len(y_test):.1f}%)")

In [ ]:
# Standardize features (important for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Features standardized")
print(f"\nTraining set statistics (after scaling):")
print(f"  Mean: {X_train_scaled.mean(axis=0).mean():.6f}")
print(f"  Std:  {X_train_scaled.std(axis=0).mean():.6f}")

# Save scaler for future use
scaler_path = os.path.join(MODELS_DIR, 'svm_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Scaler saved to {scaler_path}")

## 5. SVM Hyperparameter Tuning with GridSearchCV

In [ ]:
# Define SVM hyperparameter grid
# C: Regularization parameter (smaller = stronger regularization)
# gamma: Kernel coefficient for 'rbf', 'poly', 'sigmoid'
# kernel: Kernel type to be used in the algorithm

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'kernel': ['linear', 'rbf', 'poly'],
    'degree': [2, 3, 4]  # Only used for poly kernel
}

print("Hyperparameter Grid:")
total_combinations = 1
for param, values in param_grid.items():
    print(f"  {param}: {values}")
    total_combinations *= len(values)
print(f"\nTotal combinations to test: {total_combinations}")
print("\nNote: This may take several minutes to complete...")

In [ ]:
# Perform GridSearchCV
print("Starting GridSearchCV for SVM hyperparameter tuning...\n")
start_time = time.time()

# Create base SVM classifier
svm_base = SVC(random_state=42, probability=True)

# Perform grid search with 5-fold cross-validation
grid_search = GridSearchCV(
    svm_base,
    param_grid,
    cv=5,  # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,  # Use all available processors
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

elapsed_time = time.time() - start_time
print(f"\n✓ GridSearchCV completed in {elapsed_time:.2f} seconds")

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

In [ ]:
# Display top 10 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
results_df = results_df[['param_C', 'param_gamma', 'param_kernel', 'param_degree', 
                          'mean_test_score', 'std_test_score', 'rank_test_score']]
results_df = results_df.sort_values('rank_test_score')

print("Top 10 Parameter Combinations:")
print(results_df.head(10).to_string(index=False))

In [ ]:
# Train final SVM model with best parameters
best_svm = grid_search.best_estimator_

print(f"Training final SVM model with best parameters...")
start_time = time.time()
best_svm.fit(X_train_scaled, y_train)
training_time = time.time() - start_time

print(f"✓ Model training completed in {training_time:.2f} seconds")

# Save the trained model
model_path = os.path.join(MODELS_DIR, 'svm_classifier.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_svm, f)
print(f"✓ Model saved to {model_path}")

## 6. Cross-Validation Analysis

In [ ]:
# Perform stratified k-fold cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_svm, X_train_scaled, y_train, cv=skf, scoring='f1_weighted')

print("Cross-Validation Results (F1-Score):")
print(f"  Fold scores: {[f'{score:.4f}' for score in cv_scores]}")
print(f"  Mean score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 7. Model Evaluation on Test Set

In [ ]:
# Make predictions on training and testing sets
y_train_pred = best_svm.predict(X_train_scaled)
y_test_pred = best_svm.predict(X_test_scaled)

# Get probability predictions for ROC curve
y_train_proba = best_svm.predict_proba(X_train_scaled)
y_test_proba = best_svm.predict_proba(X_test_scaled)

print("Predictions generated successfully")

In [ ]:
# Calculate metrics
metrics = {}

# Training metrics
metrics['train_accuracy'] = accuracy_score(y_train, y_train_pred)
metrics['train_precision'] = precision_score(y_train, y_train_pred, average='weighted')
metrics['train_recall'] = recall_score(y_train, y_train_pred, average='weighted')
metrics['train_f1'] = f1_score(y_train, y_train_pred, average='weighted')

# Testing metrics
metrics['test_accuracy'] = accuracy_score(y_test, y_test_pred)
metrics['test_precision'] = precision_score(y_test, y_test_pred, average='weighted')
metrics['test_recall'] = recall_score(y_test, y_test_pred, average='weighted')
metrics['test_f1'] = f1_score(y_test, y_test_pred, average='weighted')

print("="*80)
print("SVM MODEL EVALUATION METRICS")
print("="*80)

print("\nTraining Set Performance:")
print(f"  Accuracy:  {metrics['train_accuracy']:.4f}")
print(f"  Precision: {metrics['train_precision']:.4f}")
print(f"  Recall:    {metrics['train_recall']:.4f}")
print(f"  F1-Score:  {metrics['train_f1']:.4f}")

print("\nTesting Set Performance:")
print(f"  Accuracy:  {metrics['test_accuracy']:.4f}")
print(f"  Precision: {metrics['test_precision']:.4f}")
print(f"  Recall:    {metrics['test_recall']:.4f}")
print(f"  F1-Score:  {metrics['test_f1']:.4f}")

print(f"\nOverfitting Check:")
acc_diff = metrics['train_accuracy'] - metrics['test_accuracy']
print(f"  Accuracy difference (train - test): {acc_diff:.4f}")
if acc_diff > 0.1:
    print(f"  ⚠️  Potential overfitting detected (difference > 0.1)")
else:
    print(f"  ✓ Model shows balanced generalization")

In [ ]:
# Classification report
print("\n" + "="*80)
print("CLASSIFICATION REPORT (Testing Set)")
print("="*80)
print(classification_report(y_test, y_test_pred))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)

print("Confusion Matrix (Testing Set):")
print(cm)

## 8. Visualizations

In [ ]:
# Plot confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training confusion matrix
cm_train = confusion_matrix(y_train, y_train_pred)
sns.heatmap(cm_train, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title('Confusion Matrix - Training Set', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Testing confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1], cbar=True)
axes[1].set_title('Confusion Matrix - Testing Set', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'svm_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix plot saved")

In [ ]:
# Plot metrics comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
train_values = [metrics['train_accuracy'], metrics['train_precision'], 
                 metrics['train_recall'], metrics['train_f1']]
test_values = [metrics['test_accuracy'], metrics['test_precision'], 
                metrics['test_recall'], metrics['test_f1']]

# Flatten axes for easier iteration
axes = axes.flatten()

for idx, (ax, metric_name, train_val, test_val) in enumerate(zip(axes, metrics_names, train_values, test_values)):
    x_pos = [0, 1]
    values = [train_val, test_val]
    colors = ['#3498db', '#e74c3c']
    
    bars = ax.bar(x_pos, values, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
    ax.set_ylabel(metric_name, fontsize=11, fontweight='bold')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['Training', 'Testing'])
    ax.set_ylim([0, 1])
    ax.grid(axis='y', alpha=0.3)
    
    # Add value labels on bars
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
    
    ax.set_title(f'{metric_name} Comparison', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'svm_metrics_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

print("✓ Metrics comparison plot saved")

In [ ]:
# Plot ROC curve for binary classification
if len(np.unique(y)) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Training ROC curve
    fpr_train, tpr_train, _ = roc_curve(y_train, y_train_proba[:, 1])
    roc_auc_train = auc(fpr_train, tpr_train)
    
    axes[0].plot(fpr_train, tpr_train, color='#3498db', lw=2.5, 
                 label=f'ROC Curve (AUC = {roc_auc_train:.4f})')
    axes[0].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Random Classifier')
    axes[0].set_xlabel('False Positive Rate', fontweight='bold')
    axes[0].set_ylabel('True Positive Rate', fontweight='bold')
    axes[0].set_title('ROC Curve - Training Set', fontweight='bold')
    axes[0].legend(loc='lower right')
    axes[0].grid(alpha=0.3)
    
    # Testing ROC curve
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_proba[:, 1])
    roc_auc_test = auc(fpr_test, tpr_test)
    
    axes[1].plot(fpr_test, tpr_test, color='#e74c3c', lw=2.5, 
                 label=f'ROC Curve (AUC = {roc_auc_test:.4f})')
    axes[1].plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Random Classifier')
    axes[1].set_xlabel('False Positive Rate', fontweight='bold')
    axes[1].set_ylabel('True Positive Rate', fontweight='bold')
    axes[1].set_title('ROC Curve - Testing Set', fontweight='bold')
    axes[1].legend(loc='lower right')
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'svm_roc_curve.png'), dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✓ ROC curve plot saved")
    print(f"  Training AUC: {roc_auc_train:.4f}")
    print(f"  Testing AUC: {roc_auc_test:.4f}")
else:
    print("Note: ROC curve is for binary classification. Multiclass classification detected.")

## 9. Model Summary and Export Results

In [ ]:
# Prepare summary report
summary_report = f"""
================================================================================
SVM CLASSIFIER - MODEL SUMMARY REPORT
================================================================================

DATASET INFORMATION:
  Total Samples: {df.shape[0]}
  Features: {df.shape[1] - 1}
  Classes: {len(np.unique(y))}
  Training Samples: {X_train.shape[0]}
  Testing Samples: {X_test.shape[0]}

BEST HYPERPARAMETERS:
  Kernel: {grid_search.best_params_['kernel']}
  C: {grid_search.best_params_['C']}
  Gamma: {grid_search.best_params_['gamma']}
  Degree: {grid_search.best_params_.get('degree', 'N/A')}

TRAINING PERFORMANCE:
  Accuracy: {metrics['train_accuracy']:.4f}
  Precision: {metrics['train_precision']:.4f}
  Recall: {metrics['train_recall']:.4f}
  F1-Score: {metrics['train_f1']:.4f}

TESTING PERFORMANCE:
  Accuracy: {metrics['test_accuracy']:.4f}
  Precision: {metrics['test_precision']:.4f}
  Recall: {metrics['test_recall']:.4f}
  F1-Score: {metrics['test_f1']:.4f}

CROSS-VALIDATION (5-Fold):
  Mean F1-Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})

SAVED ARTIFACTS:
  Model: {model_path}
  Scaler: {scaler_path}
  Results Directory: {RESULTS_DIR}

================================================================================
"""

print(summary_report)

# Save report to file
report_path = os.path.join(RESULTS_DIR, 'svm_model_report.txt')
with open(report_path, 'w') as f:
    f.write(summary_report)
print(f"✓ Report saved to {report_path}")

In [ ]:
# Save detailed results to CSV
results_data = {
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    'Training': [metrics['train_accuracy'], metrics['train_precision'], 
                  metrics['train_recall'], metrics['train_f1']],
    'Testing': [metrics['test_accuracy'], metrics['test_precision'], 
                 metrics['test_recall'], metrics['test_f1']]
}

results_csv = pd.DataFrame(results_data)
results_csv_path = os.path.join(RESULTS_DIR, 'svm_performance_metrics.csv')
results_csv.to_csv(results_csv_path, index=False)

print(f"✓ Performance metrics saved to {results_csv_path}")
print("\nResults Summary:")
print(results_csv.to_string(index=False))

## 10. Model Usage Example for Future Predictions

In [ ]:
# Example: How to use the trained model for new predictions
print("\n" + "="*80)
print("MODEL USAGE EXAMPLE")
print("="*80)

example_code = """
# Load the trained model and scaler
import pickle

# Load model and scaler
with open('./models/svm_classifier.pkl', 'rb') as f:
    model = pickle.load(f)

with open('./models/svm_scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

# Prepare new data (must have same number of features)
new_data = [[feature1, feature2, ..., featureN]]

# Scale the new data using the same scaler
new_data_scaled = scaler.transform(new_data)

# Make prediction
prediction = model.predict(new_data_scaled)
probabilities = model.predict_proba(new_data_scaled)

print(f"Predicted class: {prediction[0]}")
print(f"Class probabilities: {probabilities[0]}")
"""

print(example_code)